# 5. Correlation and Covariance — Mapping How Inputs Relate

**Building a Heart Disease Risk-Screening System — Notebook 5 of 12, Stage 3: Mapping Relationships Between Signals**

With trustworthy data collection established (Notebook 4), we can finally ask how
the inputs relate to each other and to the outcome we're trying to predict. This is
the first step toward *combining* variables into a model — and the step where two
of the most common modeling mistakes (double-counting redundant inputs, trusting a
misleading single number) first become visible.

## The topic

**Correlation** measures the strength and direction of a *linear* relationship
between two variables, on a unit-free -1-to-+1 scale. **Covariance** is the same
idea in the variables' original units — rarely interpreted directly, but the
quantity correlation is built from.

## Why it matters for this system

Two failure modes live here. First: a model built on two inputs that are
correlated *with each other* (not just with the outcome) can't cleanly credit
either one — this is **multicollinearity**, and it will resurface directly when the
regression model in Notebook 9 tries to interpret coefficients. Second: a single
correlation number computed across the whole registry can hide a relationship that
reverses within subgroups (**Simpson's paradox**) — trusting the pooled number
without checking subgroups is a standing risk for any system built on this kind of
summary statistic.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("../5. MLOps/2. End-to-End ML/data/heart_disease_cleaned_2.csv", index_col=0)
print(df.shape)

## The toolkit

| Tool | Use |
|---|---|
| **Pearson correlation** | Strength/direction of a *linear* relationship between two continuous variables |
| **Correlation matrix / heatmap** | Scanning many pairs at once |
| **Scatter plot** | Checking whether a correlation number is actually telling the truth about the shape |
| **Variance Inflation Factor (VIF)** | Formally flagging multicollinearity among candidate model inputs |
| **Grouped vs. pooled correlation** | Catching Simpson's-paradox-style reversals |

## How to choose

Start with the correlation matrix to get oriented across all pairs at once, but
never stop there — always follow a notable correlation (high or surprisingly low)
with a scatter plot, since a single number can't distinguish "genuinely no linear
relationship" from "a strong but non-linear one." Run VIF specifically once you
have a shortlist of candidate model inputs, not on every column — it's a
pre-modeling check, not a general-purpose exploration tool. Check grouped-vs-pooled
correlation whenever a meaningful subgroup (like `sex`) exists and the pooled
relationship will inform a decision.

## Applied to the registry

### Which inputs correlate most with disease?

In [ ]:
numeric_cols = ["age", "trestbps", "chol", "thalach", "sex", "cp", "fbs", "restecg", "exang", "slope", "ca", "thal"]
correlations = df[numeric_cols + ["target"]].corr()["target"].drop("target").sort_values(key=abs, ascending=False)
print(correlations.round(3))

### Covariance vs. correlation — same idea, different units

In [ ]:
cov_age_bp = df[["age", "trestbps"]].cov().iloc[0, 1]
corr_age_bp = df["age"].corr(df["trestbps"])
print(f"Covariance(age, trestbps) = {cov_age_bp:.1f}  (units: years x mmHg -- not directly interpretable)")
print(f"Correlation(age, trestbps) = {corr_age_bp:.3f}  (unit-free, directly comparable to any other pair)")

### Plot before you trust the number

Compare a strong-looking correlation to a weak one, scatter plots side by side —
the picture catches what the single number can hide.

In [ ]:
strongest = correlations.abs().idxmax()
weakest = correlations.abs().idxmin()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].scatter(df[strongest], df["target"] + np.random.default_rng(0).normal(0, 0.03, len(df)), alpha=0.4, s=15)
axes[0].set_title(f"{strongest} vs target  (r={correlations[strongest]:.2f})")
axes[0].set_xlabel(strongest)

axes[1].scatter(df[weakest], df["target"] + np.random.default_rng(0).normal(0, 0.03, len(df)), alpha=0.4, s=15, color="darkorange")
axes[1].set_title(f"{weakest} vs target  (r={correlations[weakest]:.2f})")
axes[1].set_xlabel(weakest)
plt.tight_layout(); plt.show()

### The correlation matrix and multicollinearity

Two *inputs* can be highly correlated with each other, not just with the target —
a problem for any model trying to credit them separately.

In [ ]:
key_inputs = ["age", "trestbps", "chol", "thalach"]
corr_matrix = df[key_inputs].corr()

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Correlation among candidate inputs -- watch for pairs correlated with EACH OTHER")
plt.tight_layout(); plt.show()

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

X = df[key_inputs].dropna()
vif = pd.DataFrame({
    "input": X.columns,
    "VIF": [variance_inflation_factor(X.values, i) for i in range(X.shape[1])],
})
print(vif.round(2))
print("\nRule of thumb: VIF above ~5-10 signals real redundancy worth addressing before")
print("trusting individual coefficients in Notebook 9's regression model.")

### Simpson's paradox: does the relationship hold up within subgroups?

Check `age` vs. `thalach` (max heart rate naturally declines with age) — pooled
across everyone, versus separately by `sex`.

In [ ]:
pooled_corr = df["age"].corr(df["thalach"])
print(f"Pooled correlation, age vs thalach: {pooled_corr:.3f}")

by_sex_corr = df.groupby("sex").apply(lambda g: g["age"].corr(g["thalach"]), include_groups=False)
by_sex_corr.index = ["female", "male"]
print("\nWithin-group correlations:")
print(by_sex_corr.round(3))
print("\nCompare the pooled number to each subgroup's own number -- a large difference between")
print("them (especially a sign flip) is exactly the Simpson's-paradox signature this notebook")
print("exists to catch. Even without a flip, this comparison is worth running any time a pooled")
print("summary will inform a real decision.")

## Systems view — what this stage hands to the next one

We now know which inputs move with the outcome, which inputs are redundant with
each other, and whether pooled relationships hold up within subgroups. That's
exactly the pre-flight check the regression model in Notebook 9 needs — but before
building the model itself, Notebooks 6-8 add one more piece: formally testing
whether a relationship this notebook surfaced is real, or could plausibly be noise.

## Try it yourself

1. Add `slope` and `ca` to the `key_inputs` VIF table — do either of them show signs
   of redundancy with the four already checked?
2. Redo the Simpson's-paradox check using `age` vs. `chol` instead of `thalach`,
   split by `fbs` instead of `sex`.
3. Find the input with the *second*-highest absolute correlation with `target` and
   check its VIF against the top-correlated input — is it adding new information,
   or substantially redundant with the top one?